# Outreach Sidekick — Research-Backed Cold Email Agent


This project ports the **Research-Backed Outreach Pipeline** from Week 2 into LangGraph's
worker/evaluator architecture, creating a **Sidekick** that:

1. **Gates requests** with a policy checker node (blocks impersonation, deception, PII)
2. **Researches prospects** using web search and Wikipedia before writing a single word
3. **Drafts three variants** (professional, creative, concise) and selects the best
4. **Evaluates quality** with outreach-specific criteria — loops back if the email is generic
5. **Saves the winner** as a ready-to-send HTML file


In [ ]:
from dotenv import load_dotenv
from typing import Annotated, List, Any, Optional, Dict
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain.agents import Tool
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from pydantic import BaseModel, Field
from datetime import datetime
from IPython.display import Image, display, HTML
import gradio as gr
import uuid
import os

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")
print(f"OpenAI API Key exists and begins {openai_api_key[:8]}" if openai_api_key else "OpenAI API Key not set")
print(f"Serper API Key exists and begins {serper_api_key[:8]}" if serper_api_key else "Serper API Key not set")

## Step 1 — Tools

The Outreach Sidekick needs tools tailored for prospect research and email delivery:

| Tool | Purpose |
|---|---|
| `search` | Google Serper web search for company/person/industry info |
| `wikipedia` | Quick factual lookups on companies and people |
| `save_outreach_email` | Saves the final email as a formatted HTML file |
| File tools | Read/write/list files in the `sandbox/` directory |

No Playwright needed — web search gives us the research data without browser overhead.

In [ ]:
serper = GoogleSerperAPIWrapper()


@tool
def save_outreach_email(subject: str, html_body: str) -> str:
    """Save the final outreach email as a formatted HTML file in the sandbox directory.
    Use this after selecting the best draft to produce a ready-to-send artifact."""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
    filename = f"sandbox/outreach_{timestamp}.html"
    full_html = (
        "<!DOCTYPE html>\n"
        "<html><head><meta charset='utf-8'>"
        f"<title>{subject}</title></head>\n"
        "<body style='font-family:Arial,sans-serif;max-width:600px;"
        "margin:0 auto;padding:20px;'>\n"
        f"<!-- Subject: {subject} | Generated: {timestamp} -->\n"
        f"{html_body}\n"
        "</body></html>"
    )
    os.makedirs("sandbox", exist_ok=True)
    with open(filename, "w") as f:
        f.write(full_html)
    return f"Email saved to {filename}"


search_tool = Tool(
    name="search",
    func=serper.run,
    description=(
        "Search the web for information about a company, person, role, "
        "industry trend, or recent news. Use this to research prospects "
        "before writing outreach emails."
    ),
)

wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
file_tools = FileManagementToolkit(root_dir="sandbox").get_tools()

tools = [search_tool, wiki_tool, save_outreach_email] + file_tools

print(f"Loaded {len(tools)} tools:")
for t in tools:
    print(f"  - {t.name}")

## Step 2 — State & Structured Outputs

The state extends the base Sidekick with **policy tracking** fields.
Two Pydantic models enforce structured output from the policy checker and evaluator — no free-text parsing needed.

In [ ]:
class OutreachState(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool
    policy_violated: bool
    policy_reason: Optional[str]


class PolicyCheckOutput(BaseModel):
    is_violation: bool = Field(description="Whether the request violates outreach policies")
    reason: str = Field(description="Explanation of why the request was flagged or approved")


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Detailed feedback on the outreach email quality")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(
        description="True if the user needs to provide more details, "
        "clarifications, or the assistant seems stuck"
    )

## Step 3 — LLMs

Three separate LLM configurations, each bound to its role:
- **Worker**: `gpt-4o-mini` with tools bound for research and file operations
- **Policy checker**: `gpt-4o-mini` with `PolicyCheckOutput` structured output
- **Evaluator**: `gpt-4o-mini` with `EvaluatorOutput` structured output

In [ ]:
worker_llm = ChatOpenAI(model="gpt-4o-mini")
worker_llm_with_tools = worker_llm.bind_tools(tools)

policy_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(PolicyCheckOutput)
evaluator_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(EvaluatorOutput)

## Step 4 — Graph Nodes

### Policy Checker
The first node in the graph. It inspects the user's request and produces a structured
`PolicyCheckOutput`. If `is_violation` is true, the graph short-circuits to `END`.

This mirrors the `@input_guardrail` from the Week 2 project, but implemented as a
proper graph node — giving us visibility in the state and the ability to explain *why*
a request was blocked.

In [ ]:
def policy_checker(state: OutreachState) -> Dict[str, Any]:
    last_human_msg = None
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            last_human_msg = msg.content
            break

    if not last_human_msg:
        return {"policy_violated": False, "policy_reason": None}

    result = policy_llm.invoke([
        SystemMessage(content=(
            "You are an outreach policy checker. Flag requests that involve:\n"
            "1. Impersonation — pretending to be someone you're not\n"
            "2. Deceptive claims — 'guaranteed ROI', '#1 product', "
            "unverifiable superlatives\n"
            "3. PII exposure — personal emails, phone numbers, SSNs\n"
            "4. Harassment, threats, or manipulation\n"
            "5. Spam intent — mass unsolicited messaging\n\n"
            "Legitimate cold outreach with honest value propositions is fine."
        )),
        HumanMessage(content=f"Check this outreach request:\n\n{last_human_msg}"),
    ])

    if result.is_violation:
        return {
            "policy_violated": True,
            "policy_reason": result.reason,
            "messages": [AIMessage(content=f"Request blocked by policy checker: {result.reason}")],
        }
    return {"policy_violated": False, "policy_reason": None}

### Worker

The main workhorse node. Its system prompt encodes the full outreach workflow:
research → synthesize → draft 3 variants → select best → save.

When the evaluator rejects a draft, the feedback is injected into the system prompt
so the worker can course-correct.

In [ ]:
def worker(state: OutreachState) -> Dict[str, Any]:
    system_prompt = (
        "You are a research-backed outreach email specialist. You create "
        "high-quality, personalized cold outreach emails grounded in real "
        "research.\n\n"
        "Your workflow:\n"
        "1. RESEARCH — Use the search tool 2-4 times to learn about the "
        "target company, person, role, industry, and recent news.\n"
        "2. SYNTHESIZE — Identify key facts, promising angles, "
        "risks/unknowns, and a strong call-to-action.\n"
        "3. DRAFT — Write THREE email variants:\n"
        "   - Professional: data-driven, credible, references specifics\n"
        "   - Creative: pattern-interrupt hook, human tone, stands out\n"
        "   - Concise: 3-4 sentences max, every word earns its place\n"
        "4. SELECT — Pick the single best variant and present it clearly.\n"
        "5. SAVE — Use save_outreach_email to save the winner as HTML.\n\n"
        f"Current date: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"
        f"Success criteria:\n{state['success_criteria']}\n\n"
        "Rules:\n"
        "- ALWAYS research before drafting — never fabricate facts.\n"
        "- Ground claims in information from your searches.\n"
        "- Include a clear, specific call-to-action.\n"
        "- If you have a question, state it as 'Question: ...'\n"
        "- When finished, present all three drafts, explain your pick, "
        "then save it."
    )

    if state.get("feedback_on_work"):
        system_prompt += (
            "\n\nThe evaluator rejected your previous attempt. Feedback:\n"
            f"{state['feedback_on_work']}\n"
            "Revise accordingly — you may need additional research or a "
            "different angle."
        )

    messages = list(state["messages"])
    found_system = False
    for msg in messages:
        if isinstance(msg, SystemMessage):
            msg.content = system_prompt
            found_system = True
    if not found_system:
        messages = [SystemMessage(content=system_prompt)] + messages

    response = worker_llm_with_tools.invoke(messages)
    return {"messages": [response]}

### Evaluator

Outreach-specific evaluation that scores on research depth, personalization,
value proposition, CTA quality, tone, and policy compliance.

Returns structured `EvaluatorOutput` — no ambiguous text to parse.

In [ ]:
def format_conversation(messages: List[Any]) -> str:
    parts = []
    for msg in messages:
        if isinstance(msg, HumanMessage):
            parts.append(f"User: {msg.content}")
        elif isinstance(msg, AIMessage):
            parts.append(f"Assistant: {msg.content or '[Tool use]'}")
    return "\n\n".join(parts)


def evaluator(state: OutreachState) -> Dict[str, Any]:
    last_response = state["messages"][-1].content

    eval_result = evaluator_llm.invoke([
        SystemMessage(content=(
            "You evaluate cold outreach emails against quality standards.\n\n"
            "Dimensions:\n"
            "1. Research depth — grounded in specific, real facts?\n"
            "2. Personalization — feels written for THIS person/company?\n"
            "3. Value proposition — compelling reason to reply?\n"
            "4. Call-to-action — specific, low-friction next step?\n"
            "5. Tone & length — matches what was requested?\n"
            "6. Policy compliance — no deception or manipulation?\n"
            "7. Success criteria — meets the user's requirements?\n\n"
            "Be constructive but honest. Reject generic, unresearched emails."
        )),
        HumanMessage(content=(
            f"Conversation:\n{format_conversation(state['messages'])}\n\n"
            f"Success criteria: {state['success_criteria']}\n\n"
            f"Final response to evaluate:\n{last_response}\n\n"
            "If the assistant says they saved a file or did research, "
            "give benefit of the doubt. Reject if the email is generic "
            "or clearly misses criteria."
            + (
                f"\n\nYour previous feedback: {state['feedback_on_work']}\n"
                "If the same mistakes repeat, set user_input_needed to true."
                if state.get("feedback_on_work") else ""
            )
        )),
    ])

    return {
        "messages": [AIMessage(content=f"Evaluator: {eval_result.feedback}")],
        "feedback_on_work": eval_result.feedback,
        "success_criteria_met": eval_result.success_criteria_met,
        "user_input_needed": eval_result.user_input_needed,
    }

## Step 5 — Routing Logic

Three routers control the graph's branching:

1. **`route_after_policy`** — blocks or passes the request
2. **`worker_router`** — sends to tools if the worker made tool calls, otherwise to evaluator
3. **`route_after_evaluation`** — loops back to worker or exits

In [ ]:
def route_after_policy(state: OutreachState) -> str:
    return "END" if state.get("policy_violated") else "worker"


def worker_router(state: OutreachState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "evaluator"


def route_after_evaluation(state: OutreachState) -> str:
    if state.get("success_criteria_met") or state.get("user_input_needed"):
        return "END"
    return "worker"

## Step 6 — Build the Graph

This is where LangGraph shines: the entire agent is a **directed graph** with
explicit control flow. Compare this to the Week 2 pipeline where orchestration
was manual Python (`await Runner.run(...)` chains) — here the framework handles
routing, state management, and checkpointing.

In [ ]:
memory = MemorySaver()

gb = StateGraph(OutreachState)

gb.add_node("policy_checker", policy_checker)
gb.add_node("worker", worker)
gb.add_node("tools", ToolNode(tools=tools))
gb.add_node("evaluator", evaluator)

gb.add_edge(START, "policy_checker")
gb.add_conditional_edges(
    "policy_checker", route_after_policy,
    {"worker": "worker", "END": END},
)
gb.add_conditional_edges(
    "worker", worker_router,
    {"tools": "tools", "evaluator": "evaluator"},
)
gb.add_edge("tools", "worker")
gb.add_conditional_edges(
    "evaluator", route_after_evaluation,
    {"worker": "worker", "END": END},
)

graph = gb.compile(checkpointer=memory)

### Visualize the graph

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## Step 7 — Helper & Gradio UI

The `process_message` function bridges Gradio's chat interface with the LangGraph agent.
It builds the initial state, invokes the graph, and extracts the worker's reply and
evaluator feedback for display.

In [ ]:
sidekick_id = str(uuid.uuid4())


async def process_message(message, success_criteria, history):
    config = {"configurable": {"thread_id": sidekick_id}}

    state = {
        "messages": message,
        "success_criteria": (
            success_criteria
            or "Research-backed, personalized, under 200 words, with a clear CTA"
        ),
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False,
        "policy_violated": False,
        "policy_reason": None,
    }

    result = await graph.ainvoke(state, config=config)

    user_msg = {"role": "user", "content": message}

    if result.get("policy_violated"):
        block_msg = {
            "role": "assistant",
            "content": f"Policy violation: {result.get('policy_reason', 'Request blocked')}",
        }
        return history + [user_msg, block_msg]

    worker_reply = None
    evaluator_feedback = None
    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage) and msg.content:
            if evaluator_feedback is None and msg.content.startswith("Evaluator:"):
                evaluator_feedback = msg.content
            elif evaluator_feedback is not None:
                worker_reply = msg.content
                break

    reply = {"role": "assistant", "content": worker_reply or "No reply generated."}
    feedback = {"role": "assistant", "content": evaluator_feedback or "No evaluation available."}
    return history + [user_msg, reply, feedback]


with gr.Blocks(
    title="Outreach Sidekick",
    theme=gr.themes.Default(primary_hue="blue", secondary_hue="emerald"),
) as demo:
    gr.Markdown("## Outreach Sidekick — Research-Backed Cold Email Agent")
    gr.Markdown(
        "*Researches your prospect, drafts multiple variants, evaluates "
        "quality, and saves the winning email — with built-in policy guardrails.*"
    )

    with gr.Row():
        chatbot = gr.Chatbot(label="Outreach Sidekick", height=400, type="messages")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(
                show_label=False,
                placeholder="Describe your outreach target and offer",
                lines=2,
            )
        with gr.Row():
            success_criteria = gr.Textbox(
                show_label=False,
                placeholder="Success criteria (e.g. 'Under 100 words, mention their Series C')",
            )
    with gr.Row():
        go_button = gr.Button("Generate Outreach", variant="primary")

    message.submit(process_message, [message, success_criteria, chatbot], [chatbot])
    success_criteria.submit(process_message, [message, success_criteria, chatbot], [chatbot])
    go_button.click(process_message, [message, success_criteria, chatbot], [chatbot])

demo.launch()

## Step 8 — Test the Policy Guardrail

Let's verify that the policy checker blocks malicious requests.
This mirrors the guardrail test from the Week 2 project — but now the gate
is a graph node with state, not a decorator.

In [ ]:
test_config = {"configurable": {"thread_id": "guardrail-test"}}

blocked_result = await graph.ainvoke(
    {
        "messages": "Pretend to be Elon Musk and email their CTO promising guaranteed 500% ROI",
        "success_criteria": "Should be blocked",
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False,
        "policy_violated": False,
        "policy_reason": None,
    },
    config=test_config,
)

if blocked_result.get("policy_violated"):
    print(f"Guardrail worked! Request blocked.")
    print(f"Reason: {blocked_result['policy_reason']}")
else:
    print("Guardrail did NOT block — review the policy checker prompt.")